# Language Analysis — Programming Language Distribution & Error Patterns
# Analyse des Langages — Distribution et Patterns d'Erreurs

**🇬🇧** This notebook extends the error analysis (notebooks `02` and `03`) by adding the programming language dimension. This analysis goes beyond Shimizu *et al.* (2025), who explicitly left language-specific analysis as future work.

**🇫🇷** Ce notebook étend l'analyse des erreurs (notebooks `02` et `03`) en ajoutant la dimension langage de programmation. Cette analyse va au-delà de Shimizu *et al.* (2025), qui a explicitement laissé l'analyse par langage comme perspective future.

**Languages covered:** C++, Python, Java, C, Ruby, C#, Rust, Go, Other  
**Extensions verified in dataset:** `cpp`, `py`, `java`, `c`, `rb`, `cs`, `rs`, `go`

---


**CSVs utilisés :** `data/processed/atcoder_language_distribution.csv`, `atcoder_language_by_group.csv`, `atcoder_error_by_language.csv`

**Structure :**
- **S1** : Distribution des langages par niveau de difficulté — quels langages dominent à chaque niveau / Language distribution by difficulty level
- **S2** : Préférence de langage par groupe de proficience (niveau A) — Python vs C++ vs autres selon le niveau de l'utilisateur / Language preference by group
- **S3** : Taux TLE par langage × difficulté — identifie les langages les plus pénalisés par les limites de temps / TLE rate by language × difficulty
- **S4** : Taux AC et CE par langage × difficulté — complète S3 avec les erreurs de compilation / AC and CE rates by language × difficulty


In [ ]:
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

DIFFICULTY_ORDER = ['A', 'B', 'C', 'D', 'E', 'F']
GROUP_ORDER      = ['G1', 'G2', 'G3', 'G4', 'G5', 'G6']
LANGUAGE_ORDER   = ['C++', 'Python', 'Java', 'C', 'Ruby', 'C#', 'Rust', 'Go', 'Other']

LANG_COLORS = {
    'C++':    '#2980b9',
    'Python': '#f39c12',
    'Java':   '#e74c3c',
    'C':      '#27ae60',
    'Ruby':   '#8e44ad',
    'C#':     '#16a085',
    'Rust':   '#d35400',
    'Go':     '#1abc9c',
    'Other':  '#95a5a6',
}

# Load all three pipeline outputs
df_lang     = pl.read_csv('../data/processed/atcoder_language_distribution.csv')
df_lg       = pl.read_csv('../data/processed/atcoder_language_by_group.csv')
df_err_lang = pl.read_csv('../data/processed/atcoder_error_by_language.csv')

print('Language distribution loaded:', df_lang.shape)
print('Language by group loaded:    ', df_lg.shape)
print('Error by language loaded:    ', df_err_lang.shape)


---
## Section 1 — Language Distribution by Difficulty Level
## Section 1 — Distribution des Langages par Niveau de Difficulté

**Why this figure:** The first question to answer is whether language usage shifts with problem difficulty. If harder problems require C++ (for performance), we should see Python's share shrink and C++'s grow from A to F. This is purely descriptive but sets the foundation for the error analysis in Section 3.

**Pourquoi cette figure :** La première question est de savoir si l'utilisation des langages évolue avec la difficulté. Si les problèmes difficiles favorisent C++ (pour la performance), la part de Python devrait diminuer de A à F. C'est descriptif mais pose les bases de la Section 3.

---


In [ ]:
# Pivot: difficulty × language → pct
df_pivot = (
    df_lang
    .pivot(on='language', index='difficulty', values='pct', aggregate_function='sum')
    .fill_null(0)
    .sort('difficulty')
).to_pandas().set_index('difficulty')

lang_cols = [l for l in LANGUAGE_ORDER if l in df_pivot.columns]
df_pivot = df_pivot[lang_cols]

fig, ax = plt.subplots(figsize=(13, 7))
bottom = np.zeros(len(df_pivot))

for lang in lang_cols:
    vals = df_pivot[lang].values
    bars = ax.bar(df_pivot.index, vals, bottom=bottom,
                  color=LANG_COLORS.get(lang, '#bdc3c7'),
                  label=lang, edgecolor='white', linewidth=0.5)
    for i, (bar, v) in enumerate(zip(bars, vals)):
        if v > 4:
            ax.text(bar.get_x() + bar.get_width()/2, bottom[i] + v/2,
                    f'{v:.1f}%', ha='center', va='center',
                    fontsize=8.5, fontweight='bold', color='white')
    bottom += vals

ax.set_xlabel('Difficulty Level / Niveau de difficulté', fontsize=12)
ax.set_ylabel('Share of submissions (%) / Part des soumissions (%)', fontsize=12)
ax.set_title(
    'Programming Language Distribution by Difficulty Level (A–F)\n'
    'Distribution des langages par niveau de difficulté (A–F)',
    fontsize=13, pad=12
)
ax.set_ylim(0, 105)
ax.legend(title='Language', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

print('Raw % table:')
display(df_pivot.round(1))


### Analysis / Analyse



**🇫🇷** C++ est le langage dominant sur l'ensemble des niveaux de difficulté, progressant de 51,4 % au niveau A jusqu'à 71,3 % au niveau F. Python constitue le deuxième langage le plus utilisé, avec une part relativement stable sur les niveaux A à D (autour de 28–30 %), puis une baisse nette à partir du niveau E (23,9 %) et F (17,2 %). Les autres langages, C, Java, Ruby notamment, voient leur part s'éroder progressivement : C passe de 5,6 % au niveau A à seulement 0,8 % au niveau F. C++ et Python concentrent à eux deux 88 % des soumissions aux niveaux E et F, mais cette stabilité globale masque un rééquilibrage interne significatif : la part de Python recule nettement tandis que C++ progresse en miroir. Autrement dit, la diversité linguistique ne disparaît pas au profit de deux langages équivalents, mais au profit d'un seul langage dominant, C++. Ce resserrement reflète un double effet de sélection : les utilisateurs qui tentent les problèmes les plus difficiles sont moins nombreux et plus expérimentés, et les langages interprétés ou moins performants sont progressivement abandonnés lorsque les contraintes de temps deviennent déterminantes.

**🇬🇧** C++ is the dominant language across all difficulty levels, rising from 51.4% at level A to 71.3% at level F. Python is the second most used language, with a relatively stable share from levels A to D (around 28–30%), followed by a clear decline from level E (23.9%) and F (17.2%). Other languages, C, Java, and Ruby in particular, see their share erode progressively: C drops from 5.6% at level A to just 0.8% at level F. Together, C++ and Python account for 88% of submissions at levels E and F, but this overall stability conceals a significant internal rebalancing: Python's share falls sharply while C++'s rises to compensate. In other words, linguistic diversity does not collapse in favor of two equivalent languages, but in favor of a single dominant one, C++. This concentration reflects a dual selection effect: users who attempt the hardest problems are fewer and more experienced, and interpreted or lower-performance languages are progressively abandoned when time constraints become critical.

---
## Section 2 — Language Preference by Proficiency Group (Level A)
## Section 2 — Préférence de Langage par Groupe de Proficience (Niveau A)

**Why this figure:** Level A is the only difficulty where all 6 groups (G1-G6) have data, making it the only valid comparison point across groups. The hypothesis is that beginners (G1) prefer Python, easier to learn, while experts (G6) prefer C++, faster to execute. If true, this would reveal a strong correlation between language choice and proficiency level, independent of problem difficulty.

**Pourquoi cette figure :** Le niveau A est le seul où les 6 groupes ont des données simultanément. L'hypothèse : les débutants (G1) préfèrent Python, plus facile à apprendre, et les experts (G6) préfèrent C++, plus rapide. Si confirmé, cela révèle une corrélation forte entre choix de langage et niveau de proficience, indépendamment de la difficulté.

---

In [ ]:
# Filter to level A — only level with all 6 groups
df_A = (
    df_lg
    .filter(pl.col('difficulty') == 'A')
    .pivot(on='language', index='proficiency_group', values='pct', aggregate_function='sum')
    .fill_null(0)
    .sort('proficiency_group')
).to_pandas().set_index('proficiency_group')

lang_cols_A = [l for l in LANGUAGE_ORDER if l in df_A.columns]
df_A = df_A[lang_cols_A]

fig, ax = plt.subplots(figsize=(13, 7))
bottom = np.zeros(len(df_A))

for lang in lang_cols_A:
    vals = df_A[lang].values
    bars = ax.bar(df_A.index, vals, bottom=bottom,
                  color=LANG_COLORS.get(lang, '#bdc3c7'),
                  label=lang, edgecolor='white', linewidth=0.5)
    for i, (bar, v) in enumerate(zip(bars, vals)):
        if v > 4:
            ax.text(bar.get_x() + bar.get_width()/2, bottom[i] + v/2,
                    f'{v:.1f}%', ha='center', va='center',
                    fontsize=8.5, fontweight='bold', color='white')
    bottom += vals

ax.set_xlabel('Proficiency Group / Groupe de proficience', fontsize=12)
ax.set_ylabel('Share of submissions (%) / Part des soumissions (%)', fontsize=12)
ax.set_title(
    'Language Preference by Proficiency Group — Level A Only\n'
    'Préférence de langage par groupe de proficience — Niveau A uniquement',
    fontsize=13, pad=12
)
ax.set_ylim(0, 105)
ax.legend(title='Language', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

print('Raw % table (Level A):')
display(df_A.round(1))


### Analysis / Analyse



**🇫🇷** Ce graphique remet en question l'hypothèse naïve selon laquelle les débutants utiliseraient massivement Python. En réalité, G1 soumet 54,3 % de ses soumissions en C++, contre seulement 17,4 % en Python. G1 présente également la part de C la plus élevée de tous les groupes (11,6 %), ce qui suggère que ces utilisateurs viennent majoritairement d'un cursus académique où C ou C++ sont enseignés, plutôt que d'un apprentissage autodidacte orienté Python. L'utilisation de Python suit une courbe en cloche : elle croît de G1 à G5 (pic à 34,8 %), avant de redescendre fortement chez G6 (21,0 %). Cette dynamique s'explique par la contrainte de performance : les utilisateurs intermédiaires (G3–G5) maîtrisent Python et parviennent à résoudre les problèmes jusqu'au niveau D–E avec ce langage, tandis que les experts G6, confrontés aux niveaux E–F, basculent vers C++ dont les performances sont indispensables pour passer les limites de temps.

**🇬🇧** This chart challenges the naive assumption that beginners would predominantly use Python. In reality, G1 submits 54.3% of its code in C++, compared to only 17.4% in Python. G1 also has the highest share of C of any group (11.6%), suggesting that these users mostly come from an academic background where C or C++ are taught, rather than from self-directed Python learning. Python usage follows a bell-shaped curve: it grows from G1 to G5 (peaking at 34.8%), before dropping sharply for G6 (21.0%). This dynamic is driven by performance constraints: intermediate users (G3–G5) are proficient in Python and can solve problems up to level D–E with it, while G6 experts, facing levels E–F, shift to C++ whose performance is essential for passing time limits.

---
## Section 3 — TLE Rate by Language × Difficulty
## Section 3 — Taux TLE par Langage × Difficulté

**Why this figure:** TLE is the most direct consequence of language performance. C++ compiles to native machine code; Python is interpreted and roughly 30–50x slower. On easy problems (A, B), this speed gap is irrelevant, the algorithms are simple enough that any language passes. But on hard problems (D, E, F), naive solutions hit the time limit, and the penalty is much higher for interpreted languages. This heatmap should reveal a dramatic divergence between Python and C++ TLE rates at high difficulty levels, which is the core original finding of this language analysis.

**Pourquoi cette figure :** Le TLE est la conséquence directe de la performance du langage. C++ compile en code machine natif ; Python est interprété et environ 30-50x plus lent. Sur les problèmes faciles, cet écart est sans importance. Sur les problèmes difficiles (D, E, F), les solutions naïves dépassent la limite de temps, et Python est bien plus pénalisé. Cette heatmap devrait révéler une divergence dramatique entre Python et C++, c'est le résultat original central de cette analyse.

---

In [ ]:
# Build TLE% matrix: language × difficulty
def build_error_matrix(status):
    mat = pd.DataFrame(index=LANGUAGE_ORDER, columns=DIFFICULTY_ORDER, dtype=float)
    for lang in LANGUAGE_ORDER:
        for diff in DIFFICULTY_ORDER:
            subset = df_err_lang.filter(
                (pl.col('language') == lang) &
                (pl.col('difficulty') == diff) &
                (pl.col('status_code') == status)
            )
            mat.loc[lang, diff] = subset['pct'].item() if not subset.is_empty() else np.nan
    return mat.astype(float)

tle_matrix = build_error_matrix('TLE')

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    tle_matrix,
    annot=True, fmt='.1f', cmap='Oranges',
    vmin=0, vmax=tle_matrix.max().max(),
    linewidths=0.5, linecolor='white',
    mask=tle_matrix.isna(),
    cbar_kws={'label': 'TLE rate (%)'},
    ax=ax
)
ax.set_title(
    'Time Limit Exceeded (TLE) Rate (%) by Language × Difficulty\n'
    'Taux TLE (%) par langage × niveau de difficulté',
    fontsize=13, pad=12
)
ax.set_xlabel('Difficulty Level / Niveau de difficulté', fontsize=12)
ax.set_ylabel('Language / Langage', fontsize=12)
plt.tight_layout()
plt.show()

print('TLE rate matrix (%):')
display(tle_matrix.round(1))


### Analysis / Analyse



**🇫🇷** La heatmap TLE révèle une divergence claire entre langages compilés et interprétés, mais uniquement à partir du niveau C, en dessous, les taux sont trop faibles pour être significatifs. À partir du niveau D, Python et Ruby atteignent respectivement 23,2 % et 23,6 % de TLE, soit près de trois fois le taux de C++ (8,2 %). Rust affiche des performances comparables à C++ (8,9 % au niveau D), ce qui confirme son statut de langage compilé et optimisé. C présente une particularité : son taux de TLE diminue sensiblement au niveau F (6,2 %, contre 13,0 % au niveau D). Cette baisse ne reflète pas une propriété intrinsèque du langage mais un biais de sélection, C ne représente que 0,8 % des soumissions au niveau F, ce qui signifie que seuls des utilisateurs très spécialisés s'y aventurent, mécaniquement plus performants. Ce même biais de sélection explique les excellents résultats de Rust, qui reste marginal en volume (~1 % des soumissions totales) et est utilisé quasi exclusivement par des spécialistes du langage.

**🇬🇧** The TLE heatmap reveals a clear divergence between compiled and interpreted languages, but only from level C onwards, below that, rates are too low to be meaningful. From level D, Python and Ruby reach 23.2% and 23.6% TLE respectively, roughly three times the C++ rate (8.2%). Rust performs comparably to C++ (8.9% at level D), confirming its status as a compiled and optimized language. C shows a notable pattern: its TLE rate drops substantially at level F (6.2%, down from 13.0% at level D). This decline does not reflect an intrinsic property of the language but rather a selection bias, C accounts for only 0.8% of submissions at level F, meaning only highly specialized users attempt it at that difficulty, who are mechanically more efficient. This same selection bias explains Rust's strong results: it remains marginal in volume (~1% of total submissions) and is used almost exclusively by language specialists.

---
## Section 4 — AC Rate & CE Rate by Language × Difficulty
## Section 4 — Taux AC et CE par Langage × Difficulté

**Why these two figures together:** AC rate gives the overall success picture per language. CE rate is interesting for a specific reason: Python has no separate compilation step, syntax errors are caught at runtime, and they might be classified differently in AtCoder. If Python has near-zero CE while C++ has higher CE, it would confirm that CE in competitive programming is a compiled-language phenomenon. Comparing them side by side reveals the full error landscape across languages.

**Pourquoi ces deux figures ensemble :** Le taux AC donne la vue d'ensemble du succès par langage. Le taux CE est intéressant pour une raison précise : Python n'a pas d'étape de compilation séparée, les erreurs de syntaxe sont détectées à l'exécution et peuvent être classifiées différemment. Si Python a un CE proche de zéro et C++ un CE plus élevé, cela confirmerait que le CE en programmation compétitive est un phénomène propre aux langages compilés.

---

In [ ]:
ac_matrix  = build_error_matrix('AC')
ce_matrix  = build_error_matrix('CE')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(
    ac_matrix,
    annot=True, fmt='.1f', cmap='Greens',
    vmin=0, vmax=100,
    linewidths=0.5, linecolor='white',
    mask=ac_matrix.isna(),
    cbar_kws={'label': 'AC rate (%)'},
    ax=axes[0]
)
axes[0].set_title('AC Rate (%) by Language × Difficulty\nTaux AC (%) par langage × difficulté', fontsize=12)
axes[0].set_xlabel('Difficulty Level')
axes[0].set_ylabel('Language')

sns.heatmap(
    ce_matrix,
    annot=True, fmt='.1f', cmap='Blues',
    vmin=0, vmax=ce_matrix.max().max(),
    linewidths=0.5, linecolor='white',
    mask=ce_matrix.isna(),
    cbar_kws={'label': 'CE rate (%)'},
    ax=axes[1]
)
axes[1].set_title('Compile Error (CE) Rate (%) by Language × Difficulty\nTaux CE (%) par langage × difficulté', fontsize=12)
axes[1].set_xlabel('Difficulty Level')
axes[1].set_ylabel('Language')

plt.tight_layout()
plt.show()


### Analysis / Analyse



**🇫🇷** Les taux d'AC décroissent avec la difficulté pour tous les langages, ce qui est cohérent avec les résultats des sections précédentes. Rust affiche le taux d'AC le plus élevé (84,1 % au niveau A), mais ce résultat doit être interprété avec précaution : il s'agit du langage le moins utilisé du dataset (~91 000 soumissions sur 8,8 millions), et son taux élevé reflète la sélectivité de sa communauté sur AtCoder plutôt qu'une supériorité intrinsèque. C se distingue par le taux d'AC le plus bas de tous les langages à chaque niveau, et par un taux de CE qui suit une courbe en U : très élevé au niveau A (19,6 %), probablement porté par les groupes G1–G2 qui utilisent C sans en maîtriser la syntaxe, il chute jusqu'au niveau C (9,0 %) à mesure que ces utilisateurs abandonnent le langage, puis remonte nettement jusqu'au niveau F (21,8 %), traduisant une vraie difficulté syntaxique lorsqu'il faut implémenter des algorithmes complexes en C, même pour des utilisateurs avancés. C'est le seul langage à présenter ce comportement : pour tous les autres langages compilés, le CE diminue monotonement avec la difficulté. À noter enfin que Python et Ruby ne produisent aucun CE dans le dataset, ce qui confirme que le Compile Error est structurellement un phénomène propre aux langages compilés.

**🇬🇧** AC rates decrease with difficulty for all languages, consistent with the results of previous sections. Rust shows the highest AC rate (84.1% at level A), but this must be interpreted with caution: it is the least-used language in the dataset (~91,000 submissions out of 8.8 million), and its high rate reflects the selectivity of its AtCoder community rather than any intrinsic superiority. C stands out with the lowest AC rate of all languages at every level, and with a CE rate that follows a U-shaped curve: very high at level A (19.6%), likely driven by G1–G2 users who use C without mastering its syntax, it drops to level C (9.0%) as those users abandon the language, then rises sharply back to level F (21.8%), reflecting a genuine syntactic difficulty when implementing complex algorithms in C, even for advanced users. It is the only language to exhibit this pattern: for all other compiled languages, CE decreases monotonically with difficulty. Finally, Python and Ruby produce no CE in the dataset, confirming that Compile Error is structurally a compiled-language phenomenon.